# Modul 5: Bifurkationen

---

## Qualitative Übergänge

**Bifurkation** (lat. *bifurcus* = zweigabelig): Ein Punkt, an dem sich das qualitative Verhalten eines Systems **grundlegend ändert**, wenn ein Parameter variiert wird.

Bifurkationen sind der Mechanismus, durch den:
- Fixpunkte entstehen oder verschwinden
- Stabilität wechselt
- Oszillationen geboren werden
- **Chaos aus Ordnung entsteht**

Sie sind die **Strukturbrüche** in der Landschaft dynamischer Systeme.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import fsolve

plt.rcParams.update({
    'figure.figsize': (12, 8),
    'figure.dpi': 100,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'lines.linewidth': 2,
})

%matplotlib inline

---

## 5.1 Bifurkationen in 1D: Die Grundtypen

In 1D gibt es drei fundamentale Bifurkationstypen. Jeder höherdimensionale Fall kann lokal auf diese reduziert werden.

### 5.1.1 Die Sattelknoten-Bifurkation (Fold/Tangent)

**Normalform:**
$$\dot{x} = r + x^2$$

Was passiert, wenn $r$ variiert?

In [ ]:
def saddle_node_bifurcation():
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Obere Reihe: f(x) für verschiedene r
    r_values = [-1, 0, 1]
    titles = [r'$r < 0$: Zwei Fixpunkte', r'$r = 0$: Bifurkation', r'$r > 0$: Keine Fixpunkte']
    
    x = np.linspace(-3, 3, 500)
    
    for ax, r, title in zip(axes[0], r_values, titles):
        y = r + x**2
        ax.plot(x, y, 'b-', linewidth=2.5)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.fill_between(x, y, 0, where=(y > 0), alpha=0.15, color='green')
        ax.fill_between(x, y, 0, where=(y < 0), alpha=0.15, color='red')
        
        # Fixpunkte
        if r < 0:
            x_fps = [np.sqrt(-r), -np.sqrt(-r)]
            ax.plot(x_fps[0], 0, 'ko', markersize=12)  # Instabil
            ax.plot(x_fps[1], 0, 'ko', markersize=12, fillstyle='none', markeredgewidth=2)  # Stabil (eigentlich andersrum)
        elif r == 0:
            ax.plot(0, 0, 'ko', markersize=12, fillstyle='left')  # Halb-stabil
        
        ax.set_xlabel(r'$x$')
        ax.set_ylabel(r'$\dot{x}$')
        ax.set_title(title)
        ax.set_xlim(-3, 3)
        ax.set_ylim(-2, 5)
    
    # Untere Reihe: Bifurkationsdiagramm
    ax_bif = fig.add_subplot(2, 1, 2)
    
    r = np.linspace(-2, 0, 500)
    x_stable = -np.sqrt(-r)  # Stabiler Zweig (negativer Ast, f' < 0)
    x_unstable = np.sqrt(-r)  # Instabiler Zweig (positiver Ast, f' > 0)
    
    ax_bif.plot(r, x_stable, 'b-', linewidth=3, label='Stabil')
    ax_bif.plot(r, x_unstable, 'r--', linewidth=3, label='Instabil')
    ax_bif.plot(0, 0, 'ko', markersize=15)  # Bifurkationspunkt
    
    ax_bif.axhline(y=0, color='k', linewidth=0.5)
    ax_bif.axvline(x=0, color='k', linewidth=0.5, linestyle='--')
    
    ax_bif.set_xlabel(r'Parameter $r$', fontsize=14)
    ax_bif.set_ylabel(r'Fixpunkt $x^*$', fontsize=14)
    ax_bif.set_title('Bifurkationsdiagramm: Sattelknoten-Bifurkation', fontsize=16)
    ax_bif.legend(loc='upper left', fontsize=12)
    ax_bif.set_xlim(-2, 1)
    ax_bif.set_ylim(-2, 2)
    
    ax_bif.annotate('Bifurkationspunkt\n$r_c = 0$', xy=(0, 0), xytext=(0.5, 1),
                   fontsize=12, arrowprops=dict(arrowstyle='->'))
    
    plt.tight_layout()
    return fig

saddle_node_bifurcation()
plt.show()

### Die Sattelknoten-Bifurkation: Interpretation

- **$r < 0$**: Zwei Fixpunkte existieren (ein stabiler, ein instabiler)
- **$r = 0$**: Die Fixpunkte **kollidieren** und bilden einen halb-stabilen Punkt
- **$r > 0$**: Beide Fixpunkte sind **verschwunden** ("annihiliert")

**Physikalische Bedeutung:** Das System verliert plötzlich sein Gleichgewicht. Beispiele:
- Kipppunkte im Klima
- Plötzlicher Kollaps von Populationen
- Laser-Schwelle

### 5.1.2 Die Transkritische Bifurkation

**Normalform:**
$$\dot{x} = rx - x^2$$

In [ ]:
def transcritical_bifurcation():
    fig = plt.figure(figsize=(14, 6))
    
    # Bifurkationsdiagramm
    ax = fig.add_subplot(1, 1, 1)
    
    r = np.linspace(-2, 2, 500)
    
    # Fixpunkt x* = 0 (immer vorhanden)
    ax.plot(r, np.zeros_like(r), 'b-', linewidth=3, label=r'$x^* = 0$')
    
    # Fixpunkt x* = r
    ax.plot(r, r, 'b-', linewidth=3, label=r'$x^* = r$')
    
    # Stabilitätsmarkierung: stabil als durchgezogen, instabil als gestrichelt
    # x* = 0: stabil für r < 0, instabil für r > 0
    ax.plot(r[r < 0], np.zeros_like(r[r < 0]), 'b-', linewidth=4)
    ax.plot(r[r > 0], np.zeros_like(r[r > 0]), 'r--', linewidth=3)
    
    # x* = r: instabil für r < 0, stabil für r > 0
    ax.plot(r[r < 0], r[r < 0], 'r--', linewidth=3)
    ax.plot(r[r > 0], r[r > 0], 'b-', linewidth=4)
    
    ax.plot(0, 0, 'ko', markersize=15)  # Bifurkationspunkt
    
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5, linestyle='--')
    
    ax.set_xlabel(r'Parameter $r$', fontsize=14)
    ax.set_ylabel(r'Fixpunkt $x^*$', fontsize=14)
    ax.set_title('Transkritische Bifurkation: Stabilitätstausch', fontsize=16)
    
    # Legende manuell
    ax.plot([], [], 'b-', linewidth=3, label='Stabil')
    ax.plot([], [], 'r--', linewidth=3, label='Instabil')
    ax.legend(loc='upper left', fontsize=12)
    
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    
    ax.annotate('Stabilitätstausch', xy=(0, 0), xytext=(0.8, -1.2),
               fontsize=12, arrowprops=dict(arrowstyle='->'))
    
    return fig

transcritical_bifurcation()
plt.show()

### Die Transkritische Bifurkation: Interpretation

- Beide Fixpunkte existieren immer ($x^* = 0$ und $x^* = r$)
- Bei $r = 0$ **tauschen sie ihre Stabilität**
- Typisch für Systeme mit einem "trivialen" Gleichgewicht (z.B. Aussterben bei $x = 0$)

**Beispiel:** Populationsdynamik mit Allee-Effekt

### 5.1.3 Die Heugabel-Bifurkation (Pitchfork)

**Normalform (superkritisch):**
$$\dot{x} = rx - x^3$$

In [ ]:
def pitchfork_bifurcation():
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Superkritische Heugabel
    ax = axes[0]
    r = np.linspace(-1, 2, 500)
    r_pos = r[r > 0]
    
    # x* = 0
    ax.plot(r[r <= 0], np.zeros_like(r[r <= 0]), 'b-', linewidth=4)  # Stabil für r < 0
    ax.plot(r[r > 0], np.zeros_like(r[r > 0]), 'r--', linewidth=3)   # Instabil für r > 0
    
    # x* = ±√r für r > 0
    ax.plot(r_pos, np.sqrt(r_pos), 'b-', linewidth=4)   # Stabil
    ax.plot(r_pos, -np.sqrt(r_pos), 'b-', linewidth=4)  # Stabil
    
    ax.plot(0, 0, 'ko', markersize=15)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5, linestyle='--')
    
    ax.set_xlabel(r'$r$', fontsize=14)
    ax.set_ylabel(r'$x^*$', fontsize=14)
    ax.set_title('Superkritische Heugabel: $\\dot{x} = rx - x^3$', fontsize=14)
    ax.set_xlim(-1, 2)
    ax.set_ylim(-2, 2)
    ax.text(1.5, 1.0, 'Symmetrie-\nbrechung', fontsize=11, ha='center')
    
    # Subkritische Heugabel: ẋ = rx + x³
    ax = axes[1]
    r = np.linspace(-2, 1, 500)
    r_neg = r[r < 0]
    
    # x* = 0
    ax.plot(r[r >= 0], np.zeros_like(r[r >= 0]), 'r--', linewidth=3)  # Instabil für r > 0
    ax.plot(r[r < 0], np.zeros_like(r[r < 0]), 'b-', linewidth=4)    # Stabil für r < 0
    
    # x* = ±√(-r) für r < 0
    ax.plot(r_neg, np.sqrt(-r_neg), 'r--', linewidth=3)   # Instabil
    ax.plot(r_neg, -np.sqrt(-r_neg), 'r--', linewidth=3)  # Instabil
    
    ax.plot(0, 0, 'ko', markersize=15)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5, linestyle='--')
    
    ax.set_xlabel(r'$r$', fontsize=14)
    ax.set_ylabel(r'$x^*$', fontsize=14)
    ax.set_title('Subkritische Heugabel: $\\dot{x} = rx + x^3$', fontsize=14)
    ax.set_xlim(-2, 1)
    ax.set_ylim(-2, 2)
    ax.text(-1.5, 1.0, 'Instabile\nZweige', fontsize=11, ha='center', color='red')
    
    # Legende
    for ax in axes:
        ax.plot([], [], 'b-', linewidth=3, label='Stabil')
        ax.plot([], [], 'r--', linewidth=3, label='Instabil')
        ax.legend(loc='upper left')
    
    plt.tight_layout()
    return fig

pitchfork_bifurcation()
plt.show()

### Die Heugabel-Bifurkation: Symmetriebrechung

**Superkritisch ($rx - x^3$):**
- Für $r < 0$: Ein stabiler Fixpunkt bei $x = 0$
- Für $r > 0$: Der Ursprung wird instabil, zwei neue stabile Fixpunkte $x = \pm\sqrt{r}$ entstehen
- Das System muss "wählen" - **Symmetriebrechung**

**Subkritisch ($rx + x^3$):**
- Die neuen Zweige sind *instabil*
- Bei der Bifurkation springt das System **katastrophal** zu einem weit entfernten Zustand
- Keine "sanfte" Übergang

**Beispiel Superkritisch:** Euler-Knickstab, Magnetisierung bei Phasenübergang, Konvektionszellen

---

## 5.2 Die Hopf-Bifurkation: Geburt von Oszillationen

Die wichtigste Bifurkation in 2D!

**Situation:** Ein Fixpunkt hat konjugiert komplexe Eigenwerte $\lambda = \alpha(r) \pm i\omega(r)$.

Bei $r = r_c$ kreuzt $\alpha$ die Null:
- $\alpha < 0$ für $r < r_c$: stabiler Fokus
- $\alpha > 0$ für $r > r_c$: instabiler Fokus

**Was passiert bei $r = r_c$?** Ein **Grenzzyklus** wird geboren!

In [ ]:
# Superkritische Hopf-Bifurkation
# Normalform in Polarkoordinaten: ṙ = r(μ - r²), θ̇ = ω
# In kartesischen: ẋ = μx - ωy - x(x² + y²), ẏ = ωx + μy - y(x² + y²)

def hopf_system(state, t, mu, omega=1.0):
    x, y = state
    r_sq = x**2 + y**2
    return [mu * x - omega * y - x * r_sq,
            omega * x + mu * y - y * r_sq]

def hopf_bifurcation_demo():
    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    
    mu_values = [-0.5, 0, 0.5]
    titles = [r'$\mu < 0$: Stabile Spirale', r'$\mu = 0$: Bifurkation', r'$\mu > 0$: Grenzzyklus']
    
    omega = 1.0
    
    for i, (ax_phase, ax_time, mu, title) in enumerate(zip(axes[0], axes[1], mu_values, titles)):
        # Phasenportrait
        t = np.linspace(0, 50, 2000)
        
        # Von innen
        sol_in = odeint(hopf_system, [0.1, 0], t, args=(mu, omega))
        ax_phase.plot(sol_in[:, 0], sol_in[:, 1], 'b-', alpha=0.8, linewidth=1)
        
        # Von außen
        sol_out = odeint(hopf_system, [2, 0], t, args=(mu, omega))
        ax_phase.plot(sol_out[:, 0], sol_out[:, 1], 'r-', alpha=0.8, linewidth=1)
        
        # Fixpunkt
        ax_phase.plot(0, 0, 'ko', markersize=10)
        
        # Grenzzyklus wenn μ > 0
        if mu > 0:
            r_lc = np.sqrt(mu)
            theta = np.linspace(0, 2*np.pi, 100)
            ax_phase.plot(r_lc * np.cos(theta), r_lc * np.sin(theta), 'g-', 
                         linewidth=3, label=f'Grenzzyklus $r = {r_lc:.2f}$')
            ax_phase.legend()
        
        ax_phase.set_xlabel(r'$x$')
        ax_phase.set_ylabel(r'$y$')
        ax_phase.set_title(title)
        ax_phase.set_xlim(-2.5, 2.5)
        ax_phase.set_ylim(-2.5, 2.5)
        ax_phase.set_aspect('equal')
        
        # Zeitverlauf
        ax_time.plot(t, sol_in[:, 0], 'b-', linewidth=1.5, label='Von innen')
        ax_time.plot(t, sol_out[:, 0], 'r-', linewidth=1.5, label='Von außen')
        ax_time.set_xlabel(r'$t$')
        ax_time.set_ylabel(r'$x(t)$')
        ax_time.legend()
        ax_time.set_xlim(0, 50)
    
    plt.tight_layout()
    return fig

hopf_bifurcation_demo()
plt.show()

### Die Hopf-Bifurkation: Kernaussagen

**Superkritisch:**
- Bei $\mu = 0$: Stabiler Fokus → Instabiler Fokus + stabiler Grenzzyklus
- Amplitude des Grenzzyklus wächst wie $\sqrt{\mu}$ (sanft!)
- Der Grenzzyklus "erbt" die Stabilität des Fixpunkts

**Subkritisch:**
- Ein *instabiler* Grenzzyklus schrumpft auf den Fixpunkt
- Bei der Bifurkation: Plötzlicher Sprung zu großamplitudigen Oszillationen
- **Hysterese** möglich

Die Hopf-Bifurkation ist der universelle Mechanismus für das Einsetzen von Oszillationen in der Natur.

In [ ]:
# Hopf-Bifurkationsdiagramm

def hopf_bifurcation_diagram():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    mu = np.linspace(-1, 1, 500)
    
    # Fixpunkt x* = 0 (Amplitude)
    ax.plot(mu[mu <= 0], np.zeros_like(mu[mu <= 0]), 'b-', linewidth=4, label='Fixpunkt (stabil)')
    ax.plot(mu[mu > 0], np.zeros_like(mu[mu > 0]), 'r--', linewidth=3, label='Fixpunkt (instabil)')
    
    # Grenzzyklus-Amplitude = √μ
    mu_pos = mu[mu > 0]
    amplitude = np.sqrt(mu_pos)
    ax.plot(mu_pos, amplitude, 'g-', linewidth=4, label='Grenzzyklus (stabil)')
    ax.plot(mu_pos, -amplitude, 'g-', linewidth=4)
    
    ax.fill_between(mu_pos, -amplitude, amplitude, alpha=0.1, color='green')
    
    ax.plot(0, 0, 'ko', markersize=15)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5, linestyle='--')
    
    ax.set_xlabel(r'Parameter $\mu$', fontsize=14)
    ax.set_ylabel(r'Amplitude', fontsize=14)
    ax.set_title('Hopf-Bifurkationsdiagramm (superkritisch)', fontsize=16)
    ax.legend(loc='upper left', fontsize=12)
    
    ax.annotate('Hopf-Bifurkation', xy=(0, 0), xytext=(0.3, 0.6),
               fontsize=12, arrowprops=dict(arrowstyle='->'))
    
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1.5, 1.5)
    
    return fig

hopf_bifurcation_diagram()
plt.show()

---

## 5.3 Periodenverdopplung: Die Route ins Chaos

Bei diskreten Abbildungen $x_{n+1} = f(x_n)$ gibt es eine faszinierende Route zum Chaos:

**Periodenverdopplungs-Kaskade:**
1. Stabiler Fixpunkt
2. 2-Zyklus (Periode 2)
3. 4-Zyklus
4. 8-Zyklus
5. ... 2ⁿ-Zyklus ...
6. **Chaos**

Die logistische Abbildung ist das Paradebeispiel.

In [ ]:
# Die logistische Abbildung: x_{n+1} = r·x_n·(1 - x_n)

def logistic_map(x, r):
    return r * x * (1 - x)

def bifurcation_diagram_logistic():
    fig, ax = plt.subplots(figsize=(16, 10))
    
    r_min, r_max = 2.5, 4.0
    r_values = np.linspace(r_min, r_max, 2000)
    
    iterations = 1000  # Transient
    last = 100  # Punkte zum Plotten
    
    x = 0.5 * np.ones(len(r_values))  # Anfangswert
    
    # Transient überspringen
    for i in range(iterations):
        x = logistic_map(x, r_values)
    
    # Punkte sammeln
    for i in range(last):
        x = logistic_map(x, r_values)
        ax.plot(r_values, x, 'k,', markersize=0.5)
    
    # Wichtige Werte markieren
    ax.axvline(x=3.0, color='blue', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=3.449, color='blue', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=3.544, color='blue', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=3.5699, color='red', linestyle='--', alpha=0.7, linewidth=2)
    
    ax.set_xlabel(r'Parameter $r$', fontsize=14)
    ax.set_ylabel(r'$x^*$ (Attraktor)', fontsize=14)
    ax.set_title('Bifurkationsdiagramm der logistischen Abbildung', fontsize=16)
    ax.set_xlim(r_min, r_max)
    ax.set_ylim(0, 1)
    
    # Annotationen
    ax.annotate('Periode 1', xy=(2.7, 0.6), fontsize=11)
    ax.annotate('Periode 2', xy=(3.15, 0.8), fontsize=11)
    ax.annotate('Periode 4', xy=(3.46, 0.9), fontsize=11)
    ax.annotate('Chaos\n(r ≈ 3.57)', xy=(3.57, 0.1), fontsize=11, color='red')
    ax.annotate('Fenster der\nOrdnung', xy=(3.83, 0.15), fontsize=10, color='blue')
    
    return fig

bifurcation_diagram_logistic()
plt.show()

### Das Bifurkationsdiagramm: Ein Fenster in die Universalität

**Beobachtungen:**

1. **Periodenverdopplung:** Die Bifurkationen zu 2, 4, 8, 16... Zyklen häufen sich
2. **Chaos:** Ab $r \approx 3.57$ beginnt chaotisches Verhalten
3. **Fenster der Ordnung:** Mitten im Chaos gibt es Bereiche mit periodischem Verhalten (z.B. Periode-3-Fenster bei $r \approx 3.83$)

### Die Feigenbaum-Konstanten

Die Abstände zwischen den Bifurkationspunkten $r_n$ folgen einem universellen Gesetz:

$$\frac{r_n - r_{n-1}}{r_{n+1} - r_n} \to \delta \approx 4.669...$$

Diese **Feigenbaum-Konstante** $\delta$ ist **universell** - sie gilt für *alle* eindimensionalen Abbildungen mit quadratischem Maximum!

Das ist eine tiefe Strukturaussage: Die Route zum Chaos ist nicht vom Detail abhängig, sondern von der *Topologie* der Abbildung.

In [ ]:
# Die Feigenbaum-Konstante numerisch

# Bifurkationspunkte (aus Literatur)
r_bif = [3.0, 3.449490, 3.544090, 3.564407, 3.568759, 3.569692, 3.569891, 3.569934]

print("Berechnung der Feigenbaum-Konstante δ:")
print("="*50)
print(f"{'n':>3} {'r_n':>12} {'δ_n':>12}")
print("-"*50)

for n in range(len(r_bif) - 2):
    delta_n = (r_bif[n+1] - r_bif[n]) / (r_bif[n+2] - r_bif[n+1])
    print(f"{n+1:>3} {r_bif[n]:>12.6f} {delta_n:>12.6f}")

print("-"*50)
print(f"Grenzwert δ = 4.669201609...")

---

## 5.4 Bifurkationen und Physik

### Phasenübergänge

Die Heugabel-Bifurkation beschreibt **Phasenübergänge 2. Ordnung**:

- Ferromagnetismus: Paramagnet ↔ Ferromagnet
- Supraleitung: Normal ↔ Supraleiter
- Konvektion: Ruhe ↔ Konvektionsrollen (Rayleigh-Bénard)

Der Ordnungsparameter (z.B. Magnetisierung) folgt $|m| \sim \sqrt{T_c - T}$ nahe der kritischen Temperatur $T_c$.

### Kipppunkte

Die Sattelknoten-Bifurkation beschreibt **abrupte Übergänge** und **Kipppunkte**:

- Klimakipppunkte (Eisschild-Kollaps)
- Ökosystem-Zusammenbrüche
- Finanzkrisen

Charakteristisch: Vor dem Kippen verlangsamt sich die Erholung (*critical slowing down*) - ein Frühwarnsignal.

---

## 5.5 Zusammenfassung

### Die Haupttypen

| Bifurkation | Normalform | Was passiert |
|-------------|------------|-------------|
| Sattelknoten | $\dot{x} = r + x^2$ | Zwei FP entstehen/verschwinden |
| Transkritisch | $\dot{x} = rx - x^2$ | Stabilitätstausch |
| Heugabel | $\dot{x} = rx - x^3$ | Symmetriebrechung |
| Hopf | Komplexe EW kreuzen imag. Achse | Grenzzyklus entsteht |
| Periodenverdopplung | Eigenwert = -1 (diskret) | Periode verdoppelt sich |

### Kernideen

1. **Strukturelle Stabilität:** Die meisten Parameterwerte sind "generisch" - kleine Änderungen ändern nichts qualitativ.

2. **Kritische Punkte:** Bei speziellen Parameterwerten bricht die Struktur auf - Bifurkationen.

3. **Universalität:** Die lokale Struktur bei Bifurkationen ist universell (Normalformen).

4. **Route zum Chaos:** Periodenverdopplung ist ein universeller Mechanismus.

### Was kommt

Im nächsten Modul: **Chaos** - Wo die Periodenverdopplung hinführt. Das Lorenz-System, Lyapunov-Exponenten, und die Bedeutung von "sensitiver Abhängigkeit von Anfangsbedingungen".

---

## Übungen

### Übung 5.1
Analysiere die Bifurkationen in $\dot{x} = r - \cosh(x)$. Bei welchem $r$ tritt eine Bifurkation auf? Welchen Typ?

### Übung 5.2
Für $\dot{x} = rx + x^3 - x^5$: Skizziere das Bifurkationsdiagramm. Welche Bifurkationstypen treten auf?

### Übung 5.3
Zeige, dass für die logistische Abbildung $f(x) = rx(1-x)$ die erste Periodenverdopplung bei $r = 3$ auftritt. Hinweis: Der Fixpunkt wird instabil wenn $|f'(x^*)| > 1$.

In [ ]:
# Raum für deine Lösungen

